# 01 — Qwen Processor

Simplified pipeline: random topic → source search → Qwen research → source verification → narration → detailed intelligent scenes. `scenes.json` is a plain list.


In [ ]:
import os, sys, subprocess
ROOT="/content/black-history-factory"; REPO_URL="https://github.com/jonbBla/black-history-factory.git"
if not os.path.exists(ROOT): subprocess.run(["git","clone",REPO_URL,ROOT],check=True)
sys.path.insert(0,ROOT)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers","accelerate","bitsandbytes","safetensors","requests"],check=True)
# Remove cached factory modules so replaced files are actually imported.
for name in list(sys.modules):
    if name.startswith("factory"): del sys.modules[name]
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(f"[SETUP] Drive: {paths.root}")


In [ ]:
from factory.qwen_client import QwenClient
qwen=QwenClient.load("Qwen/Qwen3-4B-Instruct-2507",device="cuda",load_in_4bit=True)
print("[QWEN] Model loaded.")


In [ ]:
from factory import topic_engine,qwen_pipeline

def process_one():
    topic,jid=topic_engine.find_resumable_job(paths)
    if topic: print(f"[QWEN] RESUME {jid} | {topic.title}")
    else:
        topic,jid=topic_engine.claim_next_topic(paths)
        print(f"[QWEN] NEW {jid or '-'} | {topic.title if topic else 'No topic'}")
    if not topic: return False
    return qwen_pipeline.run_one(paths,topic,jid,config,qwen)
process_one()


In [ ]:
# Enable only after one job reaches QWEN_READY successfully.
prepared=0
while prepared < int(config.prepared_job_target):
    if not process_one(): break
    prepared += 1
    print(f"[QWEN] SESSION READY: {prepared}/{config.prepared_job_target}")
